# 12 — AIA ResNet18 Multifold Error Analysis and Research Log Update

**Purpose.** This notebook performs a CPU-only post-training error analysis for the physics-safe ResNet18 hyperparameter sensitivity experiment.

It uses the outputs already produced by Notebook 12:

- `results/metrics/aia_resnet18_hp_sensitivity_multifold_all_results.csv`
- `results/metrics/aia_resnet18_hp_sensitivity_multifold_config_summary.csv`
- `results/metrics/aia_resnet18_hp_sensitivity_multifold_selection_summary.json`
- per-config/fold prediction files:
  - `*_test_predictions.csv`
  - `*_val_predictions.csv`
  - `*_test_samples.csv`
  - `*_val_samples.csv`
  - `*_test_threshold_grid.csv`
  - `*_val_threshold_grid.csv`

**No model training is performed here.**  
This notebook should run quickly on CPU and can be run before stopping the VM.

## Key protocol reminder

Hyperparameter selection was based only on **mean validation TSS across folds**.  
Test metrics are analysed after selection and must not be used to choose a new configuration.


In [ ]:
from pathlib import Path
import json
import math
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path("/home/abmoses2000/solar_flare_aia")
METRICS_DIR = ROOT / "results" / "metrics"
FIG_DIR = ROOT / "results" / "figures"
OUT_DIR = METRICS_DIR

FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_PREFIX = "aia_resnet18_hp_sensitivity"
NB_NAME = "12_aia_resnet18_multifold_error_analysis_and_research_log_update"

print("ROOT:", ROOT)
print("METRICS_DIR exists:", METRICS_DIR.exists())
print("FIG_DIR exists:", FIG_DIR.exists())
print("Started:", datetime.now().isoformat(timespec="seconds"))


## 1. Load Notebook 12 summary outputs

In [ ]:
summary_path = METRICS_DIR / "aia_resnet18_hp_sensitivity_multifold_all_results.csv"
config_summary_path = METRICS_DIR / "aia_resnet18_hp_sensitivity_multifold_config_summary.csv"
selection_path = METRICS_DIR / "aia_resnet18_hp_sensitivity_multifold_selection_summary.json"
interpretation_path = METRICS_DIR / "aia_resnet18_hp_sensitivity_multifold_interpretation.md"

required = [summary_path, config_summary_path, selection_path]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required Notebook 12 output(s):\n" + "\n".join(missing))

all_results = pd.read_csv(summary_path)
config_summary = pd.read_csv(config_summary_path)
with open(selection_path, "r") as f:
    selection = json.load(f)

selected_config = (
    selection.get("selected_config")
    or selection.get("selected_config_by_validation")
    or selection.get("best_config")
    or "higher_dropout"
)

print("Loaded all_results:", all_results.shape)
print("Loaded config_summary:", config_summary.shape)
print("Selected config:", selected_config)

display_cols = [
    "config_name", "fold_id", "source_kind", "train_rows", "val_rows", "test_rows",
    "test_positives", "selected_threshold", "val_tss", "test_roc_auc",
    "test_pr_auc", "test_precision", "test_recall", "test_specificity", "test_tss", "test_hss",
    "test_tp", "test_tn", "test_fp", "test_fn"
]
display(all_results[[c for c in display_cols if c in all_results.columns]])


## 2. Recompute core error quantities from the official confusion matrices

In [ ]:
def safe_div(a, b):
    return np.nan if b in (0, 0.0) or pd.isna(b) else a / b

err_rows = []
for _, r in all_results.iterrows():
    tp = int(r.get("test_tp", 0))
    tn = int(r.get("test_tn", 0))
    fp = int(r.get("test_fp", 0))
    fn = int(r.get("test_fn", 0))
    pos = tp + fn
    neg = tn + fp
    total = pos + neg

    diagnostic_best = r.get("diagnostic_test_best_tss", np.nan)
    official_tss = r.get("test_tss", np.nan)

    err_rows.append({
        "config_name": r.get("config_name"),
        "source_kind": r.get("source_kind"),
        "fold_id": r.get("fold_id"),
        "train_rows": r.get("train_rows"),
        "val_rows": r.get("val_rows"),
        "test_rows": r.get("test_rows"),
        "test_positives": pos,
        "test_negatives": neg,
        "test_positive_rate": safe_div(pos, total),
        "selected_threshold": r.get("selected_threshold"),
        "test_tp": tp,
        "test_tn": tn,
        "test_fp": fp,
        "test_fn": fn,
        "false_alarm_rate": safe_div(fp, neg),
        "miss_rate": safe_div(fn, pos),
        "fp_per_tp": safe_div(fp, tp),
        "fn_per_tp": safe_div(fn, tp),
        "predicted_positive_rate": safe_div(tp + fp, total),
        "predicted_negative_rate": safe_div(tn + fn, total),
        "test_precision": r.get("test_precision"),
        "test_recall": r.get("test_recall"),
        "test_specificity": r.get("test_specificity"),
        "test_tss": official_tss,
        "test_hss": r.get("test_hss"),
        "test_roc_auc": r.get("test_roc_auc"),
        "test_pr_auc": r.get("test_pr_auc"),
        "diagnostic_test_best_tss": diagnostic_best,
        "diagnostic_minus_official_tss": diagnostic_best - official_tss,
    })

error_summary = pd.DataFrame(err_rows)

out_error_summary = OUT_DIR / "aia_resnet18_multifold_error_analysis_confusion_summary.csv"
error_summary.to_csv(out_error_summary, index=False)

print("Saved:", out_error_summary)
display(error_summary.sort_values(["config_name", "fold_id"]))


## 3. Identify fold-level failure modes

In [ ]:
fold_hardness = (
    error_summary
    .groupby("fold_id", as_index=False)
    .agg(
        n_configs=("config_name", "count"),
        mean_test_tss=("test_tss", "mean"),
        std_test_tss=("test_tss", "std"),
        mean_recall=("test_recall", "mean"),
        mean_specificity=("test_specificity", "mean"),
        mean_false_alarm_rate=("false_alarm_rate", "mean"),
        mean_miss_rate=("miss_rate", "mean"),
        mean_fp_per_tp=("fp_per_tp", "mean"),
        mean_fn_per_tp=("fn_per_tp", "mean"),
        test_positive_rate=("test_positive_rate", "mean"),
    )
    .sort_values("mean_test_tss")
)

out_fold_hardness = OUT_DIR / "aia_resnet18_multifold_error_analysis_fold_hardness.csv"
fold_hardness.to_csv(out_fold_hardness, index=False)

print("Saved:", out_fold_hardness)
display(fold_hardness)


## 4. Threshold transfer analysis: validation-selected vs test diagnostic best

In [ ]:
threshold_cols = [
    "config_name", "fold_id", "selected_threshold", "val_tss", "test_tss",
    "diagnostic_test_best_tss", "diagnostic_minus_official_tss",
    "test_precision", "test_recall", "test_specificity",
    "test_tp", "test_tn", "test_fp", "test_fn"
]

threshold_transfer = error_summary[[c for c in threshold_cols if c in error_summary.columns]].copy()
threshold_transfer = threshold_transfer.sort_values("diagnostic_minus_official_tss", ascending=False)

out_threshold_transfer = OUT_DIR / "aia_resnet18_multifold_error_analysis_threshold_transfer.csv"
threshold_transfer.to_csv(out_threshold_transfer, index=False)

print("Saved:", out_threshold_transfer)
display(threshold_transfer)


## 5. Load prediction files and label-level errors

In [ ]:
def find_prediction_bundle(config_name, fold_id, split):
    pred = METRICS_DIR / f"aia_resnet18_hp_sensitivity_{config_name}_{fold_id}_{split}_predictions.csv"
    samples = METRICS_DIR / f"aia_resnet18_hp_sensitivity_{config_name}_{fold_id}_{split}_samples.csv"
    grid = METRICS_DIR / f"aia_resnet18_hp_sensitivity_{config_name}_{fold_id}_{split}_threshold_grid.csv"

    if config_name == "baseline_existing":
        candidates_pred = [
            METRICS_DIR / f"aia_resnet18_physics_safe_multifold_fullnatural_{fold_id}_{split}_predictions.csv",
            METRICS_DIR / f"aia_resnet18_physics_safe_{fold_id}_{split}_predictions.csv",
            METRICS_DIR / f"aia_resnet18_physics_safe_fold2015_fullnatural_benchmark_fullnatural_physics_safe_{split}_predictions.csv",
        ]
        candidates_samples = [
            METRICS_DIR / f"aia_resnet18_physics_safe_multifold_fullnatural_{fold_id}_{split}_samples.csv",
            METRICS_DIR / f"aia_resnet18_physics_safe_{fold_id}_{split}_samples.csv",
            METRICS_DIR / f"aia_resnet18_physics_safe_fold2015_fullnatural_benchmark_fullnatural_physics_safe_{split}_samples.csv",
        ]
        candidates_grid = [
            METRICS_DIR / f"aia_resnet18_physics_safe_multifold_fullnatural_{fold_id}_{split}_threshold_grid.csv",
            METRICS_DIR / f"aia_resnet18_physics_safe_{fold_id}_{split}_threshold_grid.csv",
            METRICS_DIR / f"aia_resnet18_physics_safe_fold2015_fullnatural_benchmark_fullnatural_physics_safe_{split}_threshold_grid.csv",
        ]
        pred = next((p for p in candidates_pred if p.exists()), pred)
        samples = next((p for p in candidates_samples if p.exists()), samples)
        grid = next((p for p in candidates_grid if p.exists()), grid)

    return pred, samples, grid

def infer_probability_column(df):
    candidates = ["prob", "probability", "y_prob", "p", "score", "pred_prob", "prediction", "logit_sigmoid"]
    for c in candidates:
        if c in df.columns:
            return c
    numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    numeric = [c for c in numeric if c not in {"y", "label", "target", "pred", "y_pred", "index"}]
    if not numeric:
        raise ValueError("Could not infer probability column from columns: " + ", ".join(df.columns))
    return numeric[0]

def infer_label_column(df):
    candidates = ["y_true", "label", "target", "y", "label_48h_final"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError("Could not infer label column from columns: " + ", ".join(df.columns))

bundles = []
for _, r in all_results.iterrows():
    config = r["config_name"]
    fold = r["fold_id"]
    threshold = float(r["selected_threshold"])

    pred_path, sample_path, grid_path = find_prediction_bundle(config, fold, "test")
    if not pred_path.exists():
        continue

    pred_df = pd.read_csv(pred_path)
    prob_col = infer_probability_column(pred_df)

    if any(c in pred_df.columns for c in ["y_true", "label", "target", "y", "label_48h_final"]):
        label_col = infer_label_column(pred_df)
        y_true = pred_df[label_col].astype(int).values
    elif sample_path.exists():
        sample_df_tmp = pd.read_csv(sample_path)
        label_col = infer_label_column(sample_df_tmp)
        y_true = sample_df_tmp[label_col].astype(int).values
    else:
        raise ValueError(f"No label column in {pred_path} and sample missing")

    prob = pred_df[prob_col].astype(float).values
    y_pred = (prob >= threshold).astype(int)

    labelled = pred_df.copy()
    labelled["config_name"] = config
    labelled["fold_id"] = fold
    labelled["selected_threshold"] = threshold
    labelled["y_true_error_analysis"] = y_true
    labelled["prob_error_analysis"] = prob
    labelled["y_pred_error_analysis"] = y_pred
    labelled["error_type"] = np.select(
        [
            (y_true == 1) & (y_pred == 1),
            (y_true == 0) & (y_pred == 0),
            (y_true == 0) & (y_pred == 1),
            (y_true == 1) & (y_pred == 0),
        ],
        ["TP", "TN", "FP", "FN"],
        default="UNKNOWN",
    )

    if sample_path.exists():
        sample_df = pd.read_csv(sample_path)
        sample_prefixed = sample_df.add_prefix("sample_")
        labelled = pd.concat([labelled.reset_index(drop=True), sample_prefixed.reset_index(drop=True)], axis=1)

    bundles.append(labelled)

if not bundles:
    print("No prediction files found. Detailed sample-level error analysis is skipped.")
    all_labelled_errors = pd.DataFrame()
else:
    all_labelled_errors = pd.concat(bundles, ignore_index=True)
    out_labelled = OUT_DIR / "aia_resnet18_multifold_error_analysis_labelled_test_predictions.csv"
    all_labelled_errors.to_csv(out_labelled, index=False)
    print("Saved:", out_labelled)
    print("Labelled predictions:", all_labelled_errors.shape)
    display(all_labelled_errors[["config_name", "fold_id", "error_type", "y_true_error_analysis", "prob_error_analysis", "y_pred_error_analysis"]].head())


## 6. High-confidence false positives and false negatives

In [ ]:
if all_labelled_errors.empty:
    print("Skipping top-error sample export because no labelled prediction files were loaded.")
else:
    selected_errors = all_labelled_errors[all_labelled_errors["config_name"] == selected_config].copy()
    if selected_errors.empty:
        selected_errors = all_labelled_errors.copy()

    top_fp = (
        selected_errors[selected_errors["error_type"] == "FP"]
        .sort_values("prob_error_analysis", ascending=False)
        .head(100)
    )
    top_fn = (
        selected_errors[selected_errors["error_type"] == "FN"]
        .sort_values("prob_error_analysis", ascending=True)
        .head(100)
    )

    out_fp = OUT_DIR / "aia_resnet18_multifold_error_analysis_top_false_positives_selected_config.csv"
    out_fn = OUT_DIR / "aia_resnet18_multifold_error_analysis_top_false_negatives_selected_config.csv"
    top_fp.to_csv(out_fp, index=False)
    top_fn.to_csv(out_fn, index=False)

    print("Selected config for sample-level error lists:", selected_config)
    print("Saved:", out_fp, "rows:", len(top_fp))
    print("Saved:", out_fn, "rows:", len(top_fn))

    display(top_fp.head(10))
    display(top_fn.head(10))


## 7. Error visualisations

In [ ]:
import matplotlib.pyplot as plt

def save_bar(df, x, y, title, filename, rotation=30):
    fig = plt.figure(figsize=(10, 5))
    plt.bar(df[x].astype(str), df[y])
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.xticks(rotation=rotation, ha="right")
    plt.tight_layout()
    path = FIG_DIR / filename
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

cfg_tss = (
    error_summary.groupby("config_name", as_index=False)
    .agg(mean_test_tss=("test_tss", "mean"), std_test_tss=("test_tss", "std"))
    .sort_values("mean_test_tss", ascending=False)
)
save_bar(cfg_tss, "config_name", "mean_test_tss",
         "Mean official test TSS by configuration", 
         "aia_resnet18_error_analysis_mean_test_tss_by_config.png")

save_bar(fold_hardness, "fold_id", "mean_test_tss",
         "Mean official test TSS by chronological fold", 
         "aia_resnet18_error_analysis_fold_hardness_tss.png")

tmp = error_summary.copy()
tmp["config_fold"] = tmp["config_name"].astype(str) + "\n" + tmp["fold_id"].astype(str)
for metric in ["false_alarm_rate", "miss_rate", "fp_per_tp", "diagnostic_minus_official_tss"]:
    plot_df = tmp.sort_values(metric, ascending=False)
    save_bar(plot_df, "config_fold", metric,
             f"{metric} by config/fold", 
             f"aia_resnet18_error_analysis_{metric}_by_config_fold.png",
             rotation=75)


## 8. Generate concise research log update

In [ ]:
def fmt(x, nd=4):
    try:
        if pd.isna(x):
            return "NA"
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)

selected_row = config_summary.sort_values("val_tss_mean", ascending=False).iloc[0]
baseline_row = config_summary[config_summary["config_name"] == "baseline_existing"]
baseline_text = ""
if len(baseline_row):
    b = baseline_row.iloc[0]
    baseline_text = (
        f" The existing baseline had mean validation TSS={fmt(b.get('val_tss_mean'))} "
        f"and mean official test TSS={fmt(b.get('test_tss_mean'))}."
    )

hardest_fold = fold_hardness.iloc[0]
best_test_cfg = config_summary.sort_values("test_tss_mean", ascending=False).iloc[0]

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

research_log_text = f"""
## Research log update — ResNet18 physics-safe multifold hyperparameter sensitivity and error analysis

**Timestamp:** {timestamp}

### Experiment completed

Completed CPU post-analysis for `12_aia_resnet18_multifold_error_analysis_and_research_log_update.ipynb`, following the completed Notebook 12 hyperparameter sensitivity experiment.

The hyperparameter sensitivity study used a predefined, physics-safe search over:

- `lower_lr`
- `higher_dropout`
- `reduced_pos_weight`
- existing baseline comparison

Selection was based only on **mean validation TSS across chronological folds**. Test metrics were reported after selection and were not used to choose the configuration.

### Validation-selected configuration

The validation-selected configuration was **`{selected_row.get('config_name')}`**, with:

- Mean validation TSS: {fmt(selected_row.get('val_tss_mean'))} ± {fmt(selected_row.get('val_tss_std'))}
- Mean validation ROC-AUC: {fmt(selected_row.get('val_roc_auc_mean'))} ± {fmt(selected_row.get('val_roc_auc_std'))}
- Mean validation PR-AUC: {fmt(selected_row.get('val_pr_auc_mean'))} ± {fmt(selected_row.get('val_pr_auc_std'))}

Official test performance for the validation-selected configuration:

- Mean test TSS: {fmt(selected_row.get('test_tss_mean'))} ± {fmt(selected_row.get('test_tss_std'))}
- Mean test ROC-AUC: {fmt(selected_row.get('test_roc_auc_mean'))} ± {fmt(selected_row.get('test_roc_auc_std'))}
- Mean test PR-AUC: {fmt(selected_row.get('test_pr_auc_mean'))} ± {fmt(selected_row.get('test_pr_auc_std'))}
- Mean recall: {fmt(selected_row.get('test_recall_mean'))} ± {fmt(selected_row.get('test_recall_std'))}
- Mean specificity: {fmt(selected_row.get('test_specificity_mean'))} ± {fmt(selected_row.get('test_specificity_std'))}
{baseline_text}

### Error-analysis finding

The hardest chronological fold was **`{hardest_fold.get('fold_id')}`**, with mean official test TSS={fmt(hardest_fold.get('mean_test_tss'))}. This confirms that the weak fold is not fully corrected by small hyperparameter changes.

The highest mean official test TSS across configurations was observed for **`{best_test_cfg.get('config_name')}`** with mean official test TSS={fmt(best_test_cfg.get('test_tss_mean'))}. This is reported as analysis only, not as the selection criterion.

### Scientific interpretation

The controlled sensitivity study suggests that higher dropout can improve validation stability, but image-only AIA ResNet18 remains unstable across chronological folds. The weakness of the 2014 fold persists after tuning, supporting the interpretation that the limitation is not only architecture or hyperparameter choice. It is likely linked to chronological/solar-cycle phase shift and the limited information content of single-time AIA EUV cutouts.

### Next research step

Proceed to a multimodal, physics-aware stage:

1. Keep AIA imagery as a morphology/thermal-emission branch.
2. Add SHARP magnetic temporal features as a magnetic-evolution branch.
3. Include chronological/cycle-phase analysis as a regime diagnostic.
4. Preserve the official protocol: validation-selected threshold, chronological folds, no test-set tuning.

### Files generated

- `results/metrics/aia_resnet18_multifold_error_analysis_confusion_summary.csv`
- `results/metrics/aia_resnet18_multifold_error_analysis_fold_hardness.csv`
- `results/metrics/aia_resnet18_multifold_error_analysis_threshold_transfer.csv`
- `results/metrics/aia_resnet18_multifold_error_analysis_labelled_test_predictions.csv` if prediction files were available
- `results/metrics/aia_resnet18_multifold_error_analysis_top_false_positives_selected_config.csv` if prediction files were available
- `results/metrics/aia_resnet18_multifold_error_analysis_top_false_negatives_selected_config.csv` if prediction files were available
- `results/metrics/aia_resnet18_multifold_error_analysis_research_log_update.md`
- `results/figures/aia_resnet18_error_analysis_*.png`
""".strip()

out_log = OUT_DIR / "aia_resnet18_multifold_error_analysis_research_log_update.md"
out_log.write_text(research_log_text, encoding="utf-8")

print(research_log_text)
print("\nSaved:", out_log)


## 9. Final file inventory

In [ ]:
generated_patterns = [
    "aia_resnet18_multifold_error_analysis_*.csv",
    "aia_resnet18_multifold_error_analysis_*.md",
]
print("Generated metrics/log files:")
for pat in generated_patterns:
    for p in sorted(OUT_DIR.glob(pat)):
        print(" -", p, f"({p.stat().st_size/1024:.1f} KiB)")

print("\nGenerated figures:")
for p in sorted(FIG_DIR.glob("aia_resnet18_error_analysis_*.png")):
    print(" -", p, f"({p.stat().st_size/1024:.1f} KiB)")

print("\nFinished:", datetime.now().isoformat(timespec="seconds"))
